In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l1.5 Gradients and autograd

Training = nudging parameters opposite their gradient. PyTorch computes those
gradients for you: set `requires_grad`, call `.backward()`, read `.grad`.

In [ ]:
x = torch.tensor([3.0], requires_grad=True)
y = (x ** 2).sum()   # dy/dx = 2x, so at x=3 the gradient is 6
y.backward()
print("x.grad:", x.grad)

The backward pass walks the computation graph applying the chain rule. Every
training step is: forward, loss, `backward()`, optimizer step, repeat.

In [ ]:
assert torch.allclose(x.grad, torch.tensor([6.0]))